# **BE5210 Final Project - Algorithm**

##Team - *The3Neurons*

**HOW TO RUN**

1. Upload the given checkpoints.zip file (loads our model)

2. In cell number four, change this to your hidden test data :
INPUT_MAT_PATH = "truetest_data.mat"

3. In def_main(), change this variable if needed :
leaderboard_ecog = data['truetest_data']

4. PLease wait for a minute for the predictions.mat file to show up on the side.


## Importing Libraries

In [ ]:
# Importing Libraries

import numpy as np
import scipy.io
import scipy.signal
import zipfile
import torch
import torch.nn as nn
from collections import OrderedDict
import glob
import torch
from collections import OrderedDict
import glob
import time

In [ ]:
!pip install PyWavelets
import pywt

## Preprocessing Data and Running Model on Test Dataset

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, p_conv_drop=0.1):
        super().__init__()
        self.conv1d = nn.Conv1d(in_channels, out_channels, kernel_size, bias=False, padding='same')
        self.norm = nn.LayerNorm(out_channels)
        self.activation = nn.GELU()
        self.drop = nn.Dropout(p=p_conv_drop)
        self.downsample = nn.MaxPool1d(kernel_size=stride, stride=stride)
        self.stride = stride

    def forward(self, x):
        x = self.conv1d(x)
        x = torch.transpose(x, -2, -1)
        x = self.norm(x)
        x = torch.transpose(x, -2, -1)
        x = self.activation(x)
        x = self.drop(x)
        x = self.downsample(x)
        return x


class UpConvBlock(nn.Module):
    def __init__(self, scale, **args):
        super().__init__()
        self.conv_block = ConvBlock(**args)
        self.upsample = nn.Upsample(scale_factor=scale, mode='linear', align_corners=False)

    def forward(self, x):
        x = self.conv_block(x)
        x = self.upsample(x)
        return x

class AutoEncoder1D(nn.Module):
    def __init__(self, n_electrodes=64, n_freqs=40, n_channels_out=5,
                 channels=[32, 32, 64, 64, 128, 128], kernel_sizes=[7, 7, 5, 5, 5],
                 strides=[2, 2, 2, 2, 2], dilation=[1, 1, 1, 1, 1]):
        super().__init__()
        self.n_inp_features = n_freqs * n_electrodes
        self.model_depth = len(channels) - 1
        self.spatial_reduce = ConvBlock(self.n_inp_features, channels[0], kernel_size=3)
        self.downsample_blocks = nn.ModuleList([
            ConvBlock(channels[i], channels[i + 1], kernel_sizes[i], strides[i], dilation[i])
            for i in range(self.model_depth)
        ])
        channels = channels[:-1] + channels[-1:]
        self.upsample_blocks = nn.ModuleList([
            UpConvBlock(scale=strides[i],
                        in_channels=channels[i + 1] if i == self.model_depth - 1 else channels[i + 1] * 2,
                        out_channels=channels[i], kernel_size=kernel_sizes[i])
            for i in range(self.model_depth - 1, -1, -1)
        ])
        self.conv1x1_one = nn.Conv1d(channels[0] * 2, n_channels_out, kernel_size=1, padding='same')


    def forward(self, x):
        batch, elec, n_freq, time = x.shape
        x = x.reshape(batch, -1, time)
        x = self.spatial_reduce(x)
        skip_connection = []
        for i in range(self.model_depth):
            skip_connection.append(x)
            x = self.downsample_blocks[i](x)
        for i in range(self.model_depth):
            x = self.upsample_blocks[i](x)
            x = torch.cat((x, skip_connection[-1 - i]), dim=1)
        x = self.conv1x1_one(x)
        return x

In [ ]:

# Fixed file paths

## Put the test data .mat file here
INPUT_MAT_PATH = "truetest_data.mat"   ######################## change this file name to your hidden test data's file name i.e, truetest_data.mat #################
OUTPUT_MAT_PATH = "predictions.mat"
CHECKPOINT_DIR = "checkpoint_final.zip"


# Constants
WAVELET_NUM = 40
L_FREQ, H_FREQ = 40, 300
ORIGINAL_FS = 1000
DOWNSAMPLE_FS = 100
CHANNELS_NUM = [62, 48, 64]


class LiveECoGInference:
    def __init__(self, patient_id):
        self.patient_id = patient_id
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.load_model()
        self.downsample_factor = ORIGINAL_FS // DOWNSAMPLE_FS

    def load_model(self):
        # Listing all files in the zip
        with zipfile.ZipFile(CHECKPOINT_DIR, 'r') as zip_ref:


            # Finding the first file that matches the patient pattern
            for file_info in zip_ref.infolist():
                if f"model_p{self.patient_id}" in file_info.filename:
                    target_file = file_info.filename
                    with zip_ref.open(target_file) as file:
                        checkpoint = torch.load(file, map_location=self.device)
                    break
            else:
                raise FileNotFoundError(f"No checkpoint found for patient {self.patient_id}")

        print(f"Loading model for Patient {self.patient_id} from: {target_file}")

        model = AutoEncoder1D(
            n_electrodes=CHANNELS_NUM[self.patient_id - 1],
            n_freqs=WAVELET_NUM,
            n_channels_out=5
        ).to(self.device)

        state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
        fixed_state_dict = OrderedDict((k.replace("model.", ""), v) for k, v in state_dict.items())

        model.load_state_dict(fixed_state_dict)
        model.eval()
        return model

    def preprocess(self, ecog_data):

        # Bandpass filter with safe padding
        sos = scipy.signal.butter(4, [L_FREQ, H_FREQ], btype='band', fs=ORIGINAL_FS, output='sos')
        # Conservative padding
        padlen = min(27, ecog_data.shape[0] // 3)
        ecog_filtered = scipy.signal.sosfiltfilt(sos, ecog_data, axis=0, padlen=padlen)

        # Wavelet transform
        scales = pywt.scale2frequency('cmor1.5-1.0', np.arange(1, WAVELET_NUM + 1)) * ORIGINAL_FS
        scales = scales[(scales >= L_FREQ) & (scales <= H_FREQ)]

        ecog_wavelet = np.zeros((ecog_data.shape[1], WAVELET_NUM, ecog_data.shape[0]))
        for ch in range(ecog_data.shape[1]):
            coef, _ = pywt.cwt(ecog_filtered[:, ch], scales, 'cmor1.5-1.0', sampling_period=1 / ORIGINAL_FS)
            ecog_wavelet[ch, :coef.shape[0], :] = np.abs(coef)

        # Downsampling
        return scipy.signal.decimate(ecog_wavelet, self.downsample_factor, axis=-1).astype(np.float32)

    def predict(self, ecog_data):
        original_length = ecog_data.shape[0]

        # Preprocessing to 100Hz (147500 → 14750 samples)
        processed_ecog = self.preprocess(ecog_data)

        # Window parameters
        window_size = 256
        stride = 25

        # Calculating needed padding
        n_windows = (processed_ecog.shape[2] - window_size) // stride + 1
        needed_length = (n_windows - 1) * stride + window_size
        current_length = processed_ecog.shape[2]

        # Only padding if needed and with positive padding values
        if needed_length > current_length:
            padding = needed_length - current_length
            processed_ecog = np.pad(processed_ecog,
                                    ((0, 0), (0, 0), (0, max(0, padding))),
                                    mode='edge')

        # Creating windows
        n_windows = (processed_ecog.shape[2] - window_size) // stride + 1
        windows = np.zeros((n_windows, processed_ecog.shape[0], processed_ecog.shape[1], window_size))
        for i in range(n_windows):
            start = i * stride
            end = start + window_size
            windows[i] = processed_ecog[:, :, start:end]

        # Predicting in batches
        predictions = []
        batch_size = 64
        with torch.no_grad():
            for i in range(0, n_windows, batch_size):
                batch = torch.from_numpy(windows[i:i + batch_size]).float().to(self.device)
                predictions.append(self.model(batch).cpu().numpy())

        # Stitching windows with overlap-add
        output_length = (n_windows - 1) * stride + window_size
        continuous_pred = np.zeros((5, output_length))
        counts = np.zeros(output_length)

        for i, pred in enumerate(np.concatenate(predictions, axis=0)):
            start = i * stride
            end = start + window_size
            continuous_pred[:, start:end] += pred
            counts[start:end] += 1

        continuous_pred /= np.maximum(counts, 1)

        # Upsampling to 1000Hz (14750 → 147500 samples)
        upsampled = scipy.signal.resample(
            continuous_pred,
            original_length,
            axis=1
        )

        return upsampled




def main():


    # Loading input data
    data = scipy.io.loadmat(INPUT_MAT_PATH)

    # Put varibale name from inside trutest_data file
    leaderboard_ecog = data['truetest_data']    ####################### change this to truetest_data ################################



    # Processing each patient
    predicted_dg = np.zeros((3, 1), dtype=object)

    # Initializing timing variables
    total_inference_time = 0
    patient_times = []

    for patient_id in [1, 2, 3]:
        print(f"\nProcessing Patient {patient_id}...")
        # Starting timer to see how long it takes for algo to run
        start_time = time.time()

        # Getting patient data (N x channels)
        ecog_data = leaderboard_ecog[patient_id - 1, 0]

        # Dynamically setting channels based on input data
        CHANNELS_NUM[patient_id - 1] = ecog_data.shape[1]

        print(f"Input shape: {ecog_data.shape} (samples × channels)")
        print(f"Duration: {ecog_data.shape[0] / ORIGINAL_FS:.2f} seconds")




        # Running inference
        predictions = LiveECoGInference(patient_id).predict(ecog_data)
        predicted_dg[patient_id - 1, 0] = predictions.T




        # Calculating and storing timing
        patient_time = time.time() - start_time
        patient_times.append(patient_time)
        total_inference_time += patient_time

        print(f"Output shape: {predictions.T.shape}")
        print(f"Inference time: {patient_time:.2f} seconds")

    # Saving results
    scipy.io.savemat(OUTPUT_MAT_PATH, {'predicted_dg': predicted_dg})
    print(f"\nPredictions saved to {OUTPUT_MAT_PATH}")
    print("Final output structure:")
    for i in range(3):
        print(f"Patient {i + 1}: {predicted_dg[i, 0].shape} (samples × 5)")

    # Printing timing summary
    print("\nTiming Summary:")
    for i, t in enumerate(patient_times):
        print(f"Patient {i+1}: {t:.2f} seconds")
    print(f"Total inference time: {total_inference_time:.2f} seconds")

if __name__ == "__main__":
    main()

NameError: name 'scipy' is not defined

If you ran all 3 code cells correctly, it should output a predictions.mat file correctly as expected. Thanks!